<a href="https://colab.research.google.com/github/SavageLDN/refill-labels/blob/main/refill_labels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
from datetime import datetime

# Sample refill label dataset
data = [
    {"Item": "Dish Soap Refill", "Batch_ID": "DS-2026-001", "Expiry_Date": "2027-08-18", "Volume_ml": 500},
    {"Item": "Hand Wash Refill", "Batch_ID": "HW-2026-002", "Expiry_Date": "2027-08-18", "Volume_ml": 250},
    {"Item": "Surface Cleaner", "Batch_ID": "SC-2026-003", "Expiry_Date": "2028-01-01", "Volume_ml": 1000}
]

df = pd.DataFrame(data)

# Generate formatted printable label string
df["Print_Label"] = df.apply(
    lambda row: f"[{row['Batch_ID']}] {row['Item']} | {row['Volume_ml']}ml | EXP: {row['Expiry_Date']}",
    axis=1
)

print("--- Generated Labels ---")
for label in df["Print_Label"]:
    print(label)


--- Generated Labels ---
[DS-2026-001] Dish Soap Refill | 500ml | EXP: 2027-08-18
[HW-2026-002] Hand Wash Refill | 250ml | EXP: 2027-08-18
[SC-2026-003] Surface Cleaner | 1000ml | EXP: 2028-01-01


In [8]:
import io
import pandas as pd
import barcode
from barcode.writer import ImageWriter
from PIL import Image

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, Image as RLImage
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

# 1. Dataset
data = [
    {"Item": "Dish Soap Refill", "Batch_ID": "DS-2026-001", "Expiry_Date": "2027-08-18", "Volume_ml": 500},
    {"Item": "Hand Wash Refill", "Batch_ID": "HW-2026-002", "Expiry_Date": "2027-08-18", "Volume_ml": 250},
    {"Item": "Surface Cleaner", "Batch_ID": "SC-2026-003", "Expiry_Date": "2028-01-01", "Volume_ml": 1000},
    {"Item": "Laundry Detergent", "Batch_ID": "LD-2026-004", "Expiry_Date": "2027-11-30", "Volume_ml": 1500},
]
df = pd.DataFrame(data)

# 2. Function to generate a barcode image buffer in memory
def generate_barcode_buffer(code_text):
    code128 = barcode.get_barcode_class('code128')
    rv = io.BytesIO()
    # Generate barcode with minimal white margin
    barcode_instance = code128(code_text, writer=ImageWriter())
    barcode_instance.write(rv, options={"write_text": False, "quiet_zone": 2.0})
    rv.seek(0)
    return rv

# 3. Setup styles for printable label card
styles = getSampleStyleSheet()
title_style = ParagraphStyle(
    'LabelTitle',
    parent=styles['Heading3'],
    fontSize=11,
    leading=13,
    textColor=colors.HexColor("#1a1a1a"),
    spaceAfter=2
)
meta_style = ParagraphStyle(
    'LabelMeta',
    parent=styles['Normal'],
    fontSize=8,
    leading=10,
    textColor=colors.HexColor("#444444")
)

# 4. Construct individual label cards
label_cards = []
for _, row in df.iterrows():
    barcode_buf = generate_barcode_buffer(row['Batch_ID'])
    barcode_img = RLImage(barcode_buf, width=150, height=35)

    card_content = [
        Paragraph(f"<b>{row['Item']}</b>", title_style),
        Paragraph(f"Volume: <b>{row['Volume_ml']} ml</b> | Exp: <b>{row['Expiry_Date']}</b>", meta_style),
        Spacer(1, 4),
        barcode_img,
        Paragraph(f"Batch: <b>{row['Batch_ID']}</b>", meta_style)
    ]

    # Outer box styling for each individual sticker label
    card_table = Table([[card_content]], colWidths=[240])
    card_table.setStyle(TableStyle([
        ('BOX', (0, 0), (-1, -1), 0.75, colors.HexColor("#cccccc")),
        ('TOPPADDING', (0, 0), (-1, -1), 8),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 8),
        ('LEFTPADDING', (0, 0), (-1, -1), 10),
        ('RIGHTPADDING', (0, 0), (-1, -1), 10),
        ('BACKGROUND', (0, 0), (-1, -1), colors.HexColor("#fafafa"))
    ]))
    label_cards.append(card_table)

# 5. Grid layout (2 columns across A4 page)
grid_data = []
for i in range(0, len(label_cards), 2):
    row_cells = [label_cards[i]]
    if i + 1 < len(label_cards):
        row_cells.append(label_cards[i + 1])
    else:
        row_cells.append("")  # Blank spacer if odd count
    grid_data.append(row_cells)

layout_table = Table(grid_data, colWidths=[255, 255])
layout_table.setStyle(TableStyle([
    ('VALIGN', (0, 0), (-1, -1), 'TOP'),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 12),
]))

# 6. Build PDF document
pdf_filename = "refill_labels.pdf"
doc = SimpleDocTemplate(
    pdf_filename,
    pagesize=A4,
    leftMargin=20,
    rightMargin=20,
    topMargin=25,
    bottomMargin=25
)
doc.build([layout_table])
print(f"Successfully generated {pdf_filename} with {len(df)} labels.")

Successfully generated refill_labels.pdf with 4 labels.


In [9]:
pip install reportlab

In [10]:
pip install python-barcode

In [11]:
pip install pillow

In [13]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.graphics.barcode import code128
from reportlab.lib.units import mm

def create_label_pdf(dataframe, filename="refill_labels.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    width, height = letter

    y_position = height - 50 # Starting Y position
    x_position = 50 # Starting X position

    for index, row in dataframe.iterrows():
        if y_position < 100: # Check if new page is needed
            c.showPage()
            y_position = height - 50

        label_text = row["Print_Label"]
        batch_id = row["Batch_ID"]

        # Draw label text
        c.setFont('Helvetica', 12)
        c.drawString(x_position, y_position, label_text)
        y_position -= 20 # Move down for barcode

        # Generate barcode
        barcode = code128.Code128(batch_id, barHeight=10*mm, barWidth=0.3*mm)
        barcode.drawOn(c, x_position, y_position)
        y_position -= 30 # Move down for next label

        # Add a separator for clarity
        c.line(x_position, y_position, x_position + 300, y_position)
        y_position -= 10 # Space after separator

    c.save()
    print(f"PDF with labels and barcodes saved as {filename}")

# Re-generate the 'Print_Label' column for the current df
df["Print_Label"] = df.apply(
    lambda row: f"[{row['Batch_ID']}] {row['Item']} | {row['Volume_ml']}ml | EXP: {row['Expiry_Date']}",
    axis=1
)

# Generate the PDF
create_label_pdf(df)


PDF with labels and barcodes saved as refill_labels.pdf
